## Student setup

Run the modules in order: M1 creates the handoff dataset consumed by M2, M3, and M4. Keep the master dataset in approved private storage; it is intentionally excluded from this repository. Set `MEDML_MASTER_DATASET_PATH` for Module 1 and `MEDML_OUTPUT_DIR` if outputs should persist outside the repository.


# M2 | Emergency-department length of stay

This notebook follows the uploaded ED LOS analyses: describe the continuous duration, inspect its skew and operational group differences, reproduce the source analysis' 4.5-hour split, and compare simple classification and regression baselines. The target changes form, but the patient-level observation remains one ED stay.

## Clinical question, learning objectives, and source method

**Question:** How long does an ED stay last, and can information available at triage distinguish stays longer than 4.5 hours?

Students will:

- describe a right-skewed duration using robust statistics;
- explain why `> 4.5` hours is an operational teaching threshold, not a natural law;
- compare LOS across acuity, age, arrival, and prior-utilization groups;
- fit a patient-disjoint logistic baseline for the binary view and a Ridge baseline for continuous hours;
- interpret error metrics in hours and in clinical classes.

This adapts `ed_los_analysis_new_31_10.ipynb` and the accompanying preprocessing notebook to the extracted CSV and replaces hard-coded paths with a transparent local pipeline.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

import os

repo_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (
        candidate
        for candidate in repo_candidates
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir()
    ),
    Path.cwd(),
)
output_override = os.getenv("MEDML_OUTPUT_DIR")
OUTPUT_DIR = Path(output_override).expanduser() if output_override else REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = OUTPUT_DIR / "M1_dataset_for_next_module.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "M1 handoff not found. Run Module 1 first, or set MEDML_OUTPUT_DIR "
        "to the folder containing M1_dataset_for_next_module.csv."
    )
ROOT_DIR = REPO_ROOT
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Loaded latest Module 1 dataset: {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

## 1. Define the LOS analysis cohort

A duration below zero is impossible. We exclude those rows from LOS summaries and modeling, but retain the original dataset unchanged. Zero-hour stays remain in the cohort because they may represent a real same-day event or an administrative timing pattern that students should investigate rather than silently discard.

In [ ]:
negative_los = int(df["ed_los_hours"].lt(0).sum())
zero_los = int(df["ed_los_hours"].eq(0).sum())
df_los = df.loc[df["ed_los_hours"].ge(0)].copy()
df_los["los_long_4_5h"] = df_los["ed_los_hours"].gt(4.5)
print(f"Negative durations excluded: {negative_los}")
print(f"Zero-hour durations retained: {zero_los}")
print(f"LOS analysis rows: {len(df_los):,}")
print(f"Long-stay prevalence (>4.5 h): {df_los['los_long_4_5h'].mean() * 100:.2f}%")

## 2. Continuous ED LOS: center, spread, tail

The mean is useful but pulled by the long right tail. Report the median, IQR, and upper percentiles as well. The plot below is restricted to 36 hours only for visual readability; no rows are removed from `df_los` by that plotting choice.

In [ ]:
los_summary = df_los["ed_los_hours"].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99]).to_frame().T
los_summary["iqr"] = los_summary["75%"] - los_summary["25%"]
display(los_summary.round(2))
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df_los.loc[df_los["ed_los_hours"].le(36), "ed_los_hours"], bins=72, color="#ea580c", ax=axes[0])
axes[0].axvline(4.5, color="#0d9488", linestyle="--", label="4.5-hour split")
axes[0].set(title="ED LOS through 36 hours", xlabel="ED LOS (hours)", ylabel="Stays")
axes[0].legend()
sns.boxplot(x=df_los["ed_los_hours"].clip(upper=df_los["ed_los_hours"].quantile(.99)), color="#99f6e4", ax=axes[1])
axes[1].set(title="LOS with values capped at the 99th percentile for display", xlabel="ED LOS (hours)")
plt.tight_layout()
plt.show()

In [ ]:
# ED LOS is an outcome: do not impute missing values or replace valid long stays.
# Missing/negative durations remain excluded from outcome analyses; valid extremes
# are retained and flagged using the conventional 1.5 × IQR rule.

los_q1 = df_los["ed_los_hours"].quantile(0.25)
los_q3 = df_los["ed_los_hours"].quantile(0.75)
los_iqr = los_q3 - los_q1
los_lower_fence = max(0, los_q1 - 1.5 * los_iqr)
los_upper_fence = los_q3 + 1.5 * los_iqr

df_los["ed_los_iqr_outlier"] = (
    df_los["ed_los_hours"].lt(los_lower_fence)
    | df_los["ed_los_hours"].gt(los_upper_fence)
)

los_data_quality = pd.DataFrame(
    {
        "measure": [
            "Missing LOS in source data",
            "Negative LOS excluded",
            "Zero-hour LOS retained",
            "IQR upper fence (hours)",
            "Valid LOS flagged as high outlier",
        ],
        "value": [
            int(df["ed_los_hours"].isna().sum()),
            negative_los,
            zero_los,
            los_upper_fence,
            int(df_los["ed_los_iqr_outlier"].sum()),
        ],
    }
)
display(los_data_quality.round(2))

### Exercise: what does the threshold throw away?

Compare a stay at 4.49 hours with one at 4.51 hours. The binary label puts them in different groups even though their durations are almost identical. Conversely, two stays both labeled “long” can be many hours apart. This is the trade-off between an operational classification and the continuous outcome.

## 3. Group comparisons from the source analysis

The source analysis examines how LOS varies by triage and patient context. These are descriptive comparisons. They can generate hypotheses about flow and case mix, but they do not prove that acuity or arrival mode causes a longer stay.

In [ ]:
acuity_los = (
    df_los.groupby("triage_acuity", dropna=False)
    .agg(
        stays=("stay_id", "size"),
        mean_los_hours=("ed_los_hours", "mean"),
        median_los_hours=("ed_los_hours", "median"),
        q25_hours=("ed_los_hours", lambda values: values.quantile(.25)),
        q75_hours=("ed_los_hours", lambda values: values.quantile(.75)),
        long_stay_pct=("los_long_4_5h", "mean"),
    )
    .reset_index()
)
acuity_los["long_stay_pct"] *= 100
display(acuity_los.round(2))

In [ ]:
age_groups = pd.cut(df_los["age"], bins=[0, 18, 40, 65, 120], right=False, include_lowest=True)
age_los = (
    df_los.assign(age_group=age_groups)
    .groupby("age_group", observed=False)
    .agg(stays=("stay_id", "size"), median_los_hours=("ed_los_hours", "median"), long_stay_pct=("los_long_4_5h", "mean"))
    .reset_index()
)
age_los["long_stay_pct"] *= 100
arrival_los = (
    df_los.groupby("arrival_transport", dropna=False)
    .agg(stays=("stay_id", "size"), median_los_hours=("ed_los_hours", "median"), long_stay_pct=("los_long_4_5h", "mean"))
    .reset_index()
)
arrival_los["long_stay_pct"] *= 100
display(age_los.round(2))
display(arrival_los.round(2))

In [ ]:
plot_df = df_los.loc[df_los["ed_los_hours"].le(24)].copy()
plt.figure(figsize=(10, 5))
sns.boxplot(data=plot_df, x="triage_acuity", y="ed_los_hours", color="#99f6e4", showfliers=False)
plt.title("ED LOS by triage acuity, displayed through 24 hours")
plt.xlabel("Triage acuity")
plt.ylabel("ED LOS (hours)")
plt.tight_layout()
plt.show()

## 4. Define the triage-time prediction boundary

The benchmark notebooks provide a useful feature family: age and demographic context, prior ED/hospital/ICU utilization, triage measurements, chief-complaint flags, and comorbidity indicators. We use those fields here but exclude raw free text from this tabular baseline.

**Excluded:** identifiers, all timestamps, `disposition`, `ed_los`, `ed_los_hours`, outcomes, ICU timing, revisit fields, later ED measurements, and medication counts. These exclusions prevent a retrospective model from seeing the answer or information created after the intended prediction point.

In [ ]:
triage_features = [
    "age", "gender", "race", "arrival_transport", "triage_temperature_celsius_iterative",
    "triage_heartrate_iterative", "triage_resprate_iterative", "triage_o2sat_iterative", "triage_sbp_iterative",
    "triage_dbp_iterative", "triage_pain_iterative", "triage_acuity_iterative",
]
prior_features = [
    "n_ed_30d", "n_ed_90d", "n_ed_365d", "n_hosp_30d", "n_hosp_90d",
    "n_hosp_365d", "n_icu_30d", "n_icu_90d", "n_icu_365d",
]
complaint_features = [column for column in df.columns if column.startswith("chiefcom_")]
comorbidity_features = [
    column for column in df.columns
    if column.startswith("cci_") or column.startswith("eci_")
]
FEATURES = [
    column for column in triage_features + prior_features + complaint_features + comorbidity_features
    if column in df.columns
]
CATEGORICAL = [column for column in ["gender", "race", "arrival_transport"] if column in FEATURES]
NUMERIC = [column for column in FEATURES if column not in CATEGORICAL]
print(f"Predictors in the declared triage-time boundary: {len(FEATURES)}")
print("Categorical:", CATEGORICAL)
print("Numeric/binary:", len(NUMERIC))

## 5. Patient-disjoint sample and train/test split

The full cohort supports description. To keep the classroom model run responsive, the baseline uses a reproducible 50,000-row sample. `GroupShuffleSplit` assigns every stay from a sampled patient to one partition. This estimates performance for new patients in the same source, not external or temporal validity.

In [ ]:
model_df = df_los.sample(n=min(50000, len(df_los)), random_state=42).reset_index(drop=True)
X = model_df[FEATURES]
y_class = model_df["los_long_4_5h"].astype(int)
y_regression = model_df["ed_los_hours"]
groups = model_df["subject_id"]
from sklearn.model_selection import GroupShuffleSplit
splitter = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=42)
train_index, test_index = next(splitter.split(X, y_class, groups=groups))
X_train, X_test = X.iloc[train_index], X.iloc[test_index]
y_class_train, y_class_test = y_class.iloc[train_index], y_class.iloc[test_index]
y_reg_train, y_reg_test = y_regression.iloc[train_index], y_regression.iloc[test_index]
groups_train, groups_test = groups.iloc[train_index], groups.iloc[test_index]
print(f"Train rows: {len(X_train):,}; test rows: {len(X_test):,}")
print(f"Train patients: {groups_train.nunique():,}; test patients: {groups_test.nunique():,}")
print("Patient overlap:", len(set(groups_train) & set(groups_test)))
print(f"Long-stay prevalence: train={y_class_train.mean():.3f}; test={y_class_test.mean():.3f}")
assert set(groups_train).isdisjoint(set(groups_test))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_preprocessor():
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                    ("scaler", StandardScaler()),
                ]),
                NUMERIC,
            ),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", encoder),
                ]),
                CATEGORICAL,
            ),
        ],
        remainder="drop",
    )


def make_pipeline(estimator):
    return Pipeline([("preprocessor", make_preprocessor()), ("model", estimator)])

## 6. Binary baseline: long versus short stay

The prior-probability dummy tells us what can be achieved without patient information. Logistic regression is a transparent baseline after imputation, missingness indicators, scaling, and one-hot encoding. A 0.5 threshold is a starting point only; an operational threshold should be connected to a capacity or staffing action.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def classification_metrics(y_true, probabilities, threshold=0.5):
    predictions = np.asarray(probabilities) >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    has_both_classes = len(np.unique(y_true)) == 2
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predictions),
        "sensitivity": recall_score(y_true, predictions, zero_division=0),
        "specificity": tn / (tn + fp) if tn + fp else np.nan,
        "ppv": precision_score(y_true, predictions, zero_division=0),
        "npv": tn / (tn + fn) if tn + fn else np.nan,
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities) if has_both_classes else np.nan,
        "average_precision": average_precision_score(y_true, probabilities) if has_both_classes else np.nan,
        "true_positive": tp,
        "false_positive": fp,
        "true_negative": tn,
        "false_negative": fn,
    }

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

classifiers = {
    "prior dummy": DummyClassifier(strategy="prior"),
    "logistic regression": LogisticRegression(max_iter=500, class_weight="balanced", random_state=42),
    "decision tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=25, class_weight="balanced", random_state=42),
    "random forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=10,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    ),
    "gradient boosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        min_samples_leaf=15,
        random_state=42,
    ),
}

classifier_pipelines = {}
classifier_probabilities = {}
classification_rows = []
for model_name, estimator in classifiers.items():
    model = estimator if model_name == "prior dummy" else make_pipeline(estimator)
    model.fit(X_train, y_class_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    classifier_pipelines[model_name] = model
    classifier_probabilities[model_name] = probabilities
    classification_rows.append({
        "model": model_name,
        **classification_metrics(y_class_test, probabilities),
    })

classification_results = (
    pd.DataFrame(classification_rows)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)
display(classification_results.round(3))

# Keep the original logistic-regression names available for the threshold and curve cells below.
dummy = classifier_pipelines["prior dummy"]
dummy_probabilities = classifier_probabilities["prior dummy"]
logistic = classifier_pipelines["logistic regression"]
logistic_probabilities = classifier_probabilities["logistic regression"]

In [ ]:
from sklearn.metrics import roc_curve

roc_fig, roc_axis = plt.subplots(figsize=(10, 6))
for model_name, probabilities in classifier_probabilities.items():
    false_positive_rate, true_positive_rate, _ = roc_curve(y_class_test, probabilities)
    model_auc = roc_auc_score(y_class_test, probabilities)
    roc_axis.plot(
        false_positive_rate,
        true_positive_rate,
        linewidth=2,
        label=f"{model_name} (AUROC = {model_auc:.3f})",
    )

roc_axis.plot([0, 1], [0, 1], linestyle=":", color="black", label="chance (AUROC = 0.500)")
roc_axis.set(
    title="ROC curves for all classification methods",
    xlabel="False-positive rate",
    ylabel="True-positive rate / sensitivity",
    xlim=(0, 1),
    ylim=(0, 1.02),
)
roc_axis.legend(loc="lower right")
roc_axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

metric_columns = ["accuracy", "sensitivity", "specificity", "ppv", "f1", "roc_auc", "average_precision"]
classification_results.set_index("model")[metric_columns].plot(
    kind="bar",
    figsize=(13, 6),
    ylim=(0, 1),
    colormap="viridis",
)
plt.title("Combined classification metrics at threshold = 0.5")
plt.xlabel("")
plt.ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## 7. Discrimination and threshold trade-offs

ROC AUC describes ranking over all thresholds. Precision-recall behavior is especially useful when the positive class is not exactly half of the cohort. Neither curve chooses a threshold for us. The threshold table makes the operational trade-off visible: lower thresholds usually increase sensitivity but also increase alerts and false positives.

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve
fpr, tpr, _ = roc_curve(y_class_test, logistic_probabilities)
precision, recall, _ = precision_recall_curve(y_class_test, logistic_probabilities)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(fpr, tpr, color="#0d9488", label=f"AUC={roc_auc_score(y_class_test, logistic_probabilities):.3f}")
axes[0].plot([0, 1], [0, 1], ":", color="black")
axes[0].set(title="ROC curve", xlabel="False-positive rate", ylabel="Sensitivity")
axes[0].legend()
axes[1].plot(recall, precision, color="#ea580c", label=f"AP={average_precision_score(y_class_test, logistic_probabilities):.3f}")
axes[1].axhline(y_class_test.mean(), linestyle=":", color="black", label=f"prevalence={y_class_test.mean():.3f}")
axes[1].set(title="Precision-recall curve", xlabel="Sensitivity / recall", ylabel="PPV / precision")
axes[1].legend()
plt.tight_layout()
plt.show()
threshold_results = pd.DataFrame([
    {"threshold": threshold, **classification_metrics(y_class_test, logistic_probabilities, threshold)}
    for threshold in [.25, .40, .50, .60, .75]
])
display(threshold_results.round(3))

## 8. Continuous baseline: predict hours, not just a class

Ridge regression keeps the duration information. MAE is the average absolute error in hours, RMSE penalizes large misses more strongly, and $R^2$ is a variance-relative summary. A model can have a reasonable MAE while systematically underpredicting the longest stays, so inspect the residual behavior.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
ridge = make_pipeline(Ridge(alpha=1.0))
ridge.fit(X_train, y_reg_train)
regression_predictions = ridge.predict(X_test)
regression_results = pd.DataFrame([{
    "model": "ridge regression",
    "mae_hours": mean_absolute_error(y_reg_test, regression_predictions),
    "rmse_hours": np.sqrt(mean_squared_error(y_reg_test, regression_predictions)),
    "r2": r2_score(y_reg_test, regression_predictions),
}])
display(regression_results.round(3))
residuals = y_reg_test.to_numpy() - regression_predictions
residual_table = pd.DataFrame({"observed": y_reg_test.to_numpy(), "predicted": regression_predictions, "residual": residuals})
display(residual_table.describe(percentiles=[.5, .9, .99]).round(2))
plot_sample = residual_table.sample(n=min(4000, len(residual_table)), random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=plot_sample, x="observed", y="predicted", alpha=.25, color="#0d9488", ax=axes[0])
axes[0].plot([0, 36], [0, 36], "--", color="#ea580c")
axes[0].set(xlim=(0, 36), ylim=(0, 36), title="Observed and predicted LOS", xlabel="Observed hours", ylabel="Predicted hours")
sns.scatterplot(data=plot_sample, x="predicted", y="residual", alpha=.25, color="#ea580c", ax=axes[1])
axes[1].axhline(0, linestyle="--", color="black")
axes[1].set(title="Residuals", xlabel="Predicted hours", ylabel="Observed - predicted hours")
plt.tight_layout()
plt.show()

### Discussion: choose the representation

A flow manager may prefer a binary “long stay” alert, while a planning team may prefer an expected number of hours. The binary target is easier to communicate but discards duration. The regression target preserves information but its errors are harder to map to an action. State the intended action before choosing a metric.

## Student lab and handoff

1. Write one paragraph interpreting the LOS median and IQR without using the mean alone.
2. Select a threshold row and explain the practical meaning of its sensitivity, PPV, and false-positive count.
3. Identify one variable that may encode workflow rather than physiology.
4. Compare a binary and continuous formulation and state which decision each supports.

The next outcome analysis keeps the same triage boundary but changes the label. Recompute everything; metrics from LOS do not transfer to a different outcome.

In [ ]:
M2_tables = {
    "M2_los_summary.csv": los_summary,
    "M2_los_by_acuity.csv": acuity_los,
    "M2_los_by_age.csv": age_los,
    "M2_los_by_arrival.csv": arrival_los,
    "M2_classification_results.csv": classification_results,
    "M2_threshold_results.csv": threshold_results,
    "M2_regression_results.csv": regression_results,
}
for filename, table in M2_tables.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)
print(f"Wrote {len(M2_tables)} M2 tables to {OUTPUT_DIR}")